In [1]:
import numpy as np
import seaborn as sb
import pandas as pd
from RuleTree.utils.feature_utils import detect_categorical_features

c:\Users\david\miniconda3\envs\trepan-dev\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from ucimlrepo import fetch_ucirepo
heart_disease = fetch_ucirepo(id=45)
X = heart_disease.data.features
y = heart_disease.data.targets
categorical_features_names = ['sex','cp','fbs','restecg','exang','slope','thal']
categorical_features = [X.columns.get_loc(col) for col in categorical_features_names]



In [3]:
X.shape

(303, 13)

In [4]:
X.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
dtype: int64

In [5]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 30.9 KB


In [6]:
for col in X.columns:
    print(X[col].unique())

[63 67 37 41 56 62 57 53 44 52 48 54 49 64 58 60 50 66 43 40 69 59 42 55
 61 65 71 51 46 45 39 68 47 34 35 29 70 77 38 74 76]
[1 0]
[1 4 3 2]
[145 160 120 130 140 172 150 110 132 117 135 112 105 124 125 142 128 170
 155 104 180 138 108 134 122 115 118 100 200  94 165 102 152 101 126 174
 148 178 158 192 129 144 123 136 146 106 156 154 114 164]
[233 286 229 250 204 236 268 354 254 203 192 294 256 263 199 168 239 275
 266 211 283 284 224 206 219 340 226 247 167 230 335 234 177 276 353 243
 225 302 212 330 175 417 197 198 290 253 172 273 213 305 216 304 188 282
 185 232 326 231 269 267 248 360 258 308 245 270 208 264 321 274 325 235
 257 164 141 252 255 201 222 260 182 303 265 309 307 249 186 341 183 407
 217 288 220 209 227 261 174 281 221 205 240 289 318 298 564 246 322 299
 300 293 277 214 207 223 160 394 184 315 409 244 195 196 126 313 259 200
 262 215 228 193 271 210 327 149 295 306 178 237 218 242 319 166 180 311
 278 342 169 187 157 176 241 131]
[1 0]
[2 0 1]
[150 108 129 187 172 1

In [7]:
X['ca'] = X['ca'].fillna(X['ca'].mode()[0])
X['thal'] = X['thal'].fillna(X['thal'].mode()[0])

# Converti target in binario: 0 = no disease, 1 = disease
y_binary = (y > 0).astype(int)

# Usa y_binary per tutto
y = y_binary.values.ravel()  # assicurati che sia 1D


In [8]:
X.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
dtype: int64

In [9]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
from sklearn.preprocessing import StandardScaler

In [12]:
s = StandardScaler()
X_train = s.fit_transform(X_train)
X_test = s.transform(X_test)

In [13]:
# Addestra una rete (come nel paper)
mlp = MLPClassifier(
    hidden_layer_sizes=(),  # prova 0,5,10,20,40
    max_iter=500,
    random_state=42
)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)

In [14]:
print('Test Accuracy %s' % accuracy_score(y_test, y_pred))
print('Test F1-score %s' % f1_score(y_test, y_pred, average=None))
print(classification_report(y_test, y_pred))

Test Accuracy 0.8852459016393442
Test F1-score [0.88135593 0.88888889]
              precision    recall  f1-score   support

           0       0.87      0.90      0.88        29
           1       0.90      0.88      0.89        32

    accuracy                           0.89        61
   macro avg       0.88      0.89      0.89        61
weighted avg       0.89      0.89      0.89        61



In [15]:
# importa la classe direttamente dal file
from RuleTree.stumps.classification import MofNTrepanStumpClassifier
from RuleTree.tree.TrepanClassifier import TrepanClassifier
base_stumps = [MofNTrepanStumpClassifier(max_conditions=4)]
trepan_clf = TrepanClassifier(estimator = mlp, s_min=300, base_stumps=base_stumps, max_internal_nodes=10, random_state=42, categorical_features=categorical_features)
trepan_clf.fit(X_train, y_train)
y_pred_trepan = trepan_clf.predict(X_test)
print('Accuracy %s' % accuracy_score(y_test, y_pred_trepan))
print('F1-score %s' % f1_score(y_test, y_pred_trepan, average=None))
print(classification_report(y_test, y_pred_trepan))
fidelty = accuracy_score(y_pred, y_pred_trepan)
print("The fidelty respect to the oracle is %s" % fidelty)

Accuracy 0.8360655737704918
F1-score [0.83333333 0.83870968]
              precision    recall  f1-score   support

           0       0.81      0.86      0.83        29
           1       0.87      0.81      0.84        32

    accuracy                           0.84        61
   macro avg       0.84      0.84      0.84        61
weighted avg       0.84      0.84      0.84        61

The fidelty respect to the oracle is 0.8852459016393442


In [16]:
attributes = X.columns
trepan_clf.print_trepan_rules(feature_names=attributes, scaler=s)


  REGOLE GLOBALI ESTRATTE DA TREPAN
  (VALORI RISCALATI ALLA SCALA ORIGINALE)
REGOLA 1 [Nodo ID: Rlllllll]:
  IF  (2-of-{ thal == 3.0, exang == 0, ca <= 1.0 }}) AND 
      (1-of-{ ca <= 2.0 }}) AND 
      (1-of-{ oldpeak <= 3.5, sex == 0 }}) AND 
      (1-of-{ oldpeak <= 3.5 }}) AND 
      (1-of-{ oldpeak <= 1.8 }}) AND 
      (1-of-{ thal == 7.0 }}) AND 
      (3-of-{ cp == 4.0, thalach <= 165.438, fbs == 0.0 }})
  THEN Predizione = 1
  [Fedeltà: 93.33%, Copertura: 1.65%]

REGOLA 2 [Nodo ID: Rllllllr]:
  IF  (2-of-{ thal == 3.0, exang == 0, ca <= 1.0 }}) AND 
      (1-of-{ ca <= 2.0 }}) AND 
      (1-of-{ oldpeak <= 3.5, sex == 0 }}) AND 
      (1-of-{ oldpeak <= 3.5 }}) AND 
      (1-of-{ oldpeak <= 1.8 }}) AND 
      (1-of-{ thal == 7.0 }}) AND 
      (NOT (3-of-{ cp == 4.0, thalach <= 165.438, fbs == 0.0 }}))
  THEN Predizione = 0
  [Fedeltà: 93.33%, Copertura: 9.92%]

REGOLA 3 [Nodo ID: Rlllllr]:
  IF  (2-of-{ thal == 3.0, exang == 0, ca <= 1.0 }}) AND 
      (1-of-{ ca <= 2.0 }}

In [17]:
def create_mlp(hidden_units):
    if hidden_units == 0:
        return MLPClassifier(
            hidden_layer_sizes=(),
            max_iter=500,  # aumentato
            random_state=42,
            early_stopping=False,
            alpha=0.0001
        )
    else:
        return MLPClassifier(
            hidden_layer_sizes=(hidden_units,),
            max_iter=500,
            random_state=42,
            early_stopping=False,
            alpha=0.0001
        )

def select_best_hidden_units(X_train, y_train, hidden_units_list, cv_inner=5):
    """
    Seleziona il miglior numero di hidden unit con cross-validation interna.
    Come nel paper: prova {0,5,10,20,40} e sceglie il migliore.
    """
    X_train = np.asarray(X_train, dtype=np.float64)
    # y_train è già int, non convertire
    
    best_units = 10
    best_score = -1
    
    for units in hidden_units_list:
        mlp = create_mlp(units)
        scores = cross_val_score(mlp, X_train, y_train, cv=cv_inner, scoring='accuracy')
        mean_score = np.mean(scores)
        
        if mean_score > best_score:
            best_score = mean_score
            best_units = units
            
    return best_units, best_score

def train_and_evaluate_network(X_train, y_train, X_test, y_test, hidden_units):
    """
    Addestra una rete neurale e restituisce:
    - accuracy sul test set
    - modello addestrato
    - predizioni sul test set
    """
    mlp = create_mlp(hidden_units)
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc, mlp, y_pred

In [18]:
X_np = X.values.astype(np.float64)
y_np = y  # y è già numpy array (da Cell 397)
# 10-FOLD CROSS-VALIDATION (CORRETTA!)
# ============================================================
hidden_units_list = [0,5,10,20,40]
n_folds = 10
random_state = 42

accuracies_net = []
accuracies_tree = []
fidelities = []
chosen_units = []

skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)

print("\n" + "="*60)
print("INIZIO 10-FOLD CV - HEART DISEASE")
print("="*60 + "\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(X_np, y_np), 1):
    print(f"Fold {fold}/{n_folds}")
    
    # 1. Dividi i dati (USANDO NUMPY!)
    X_train = X_np[train_idx]
    y_train = y_np[train_idx]
    X_test = X_np[test_idx]
    y_test = y_np[test_idx]
    
    # 2. STANDARD SCALER (SOLO SU TRAIN!)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)  # fit su train
    X_test_scaled = scaler.transform(X_test)        # transform su test
    
    # 3. Seleziona hidden units
    best_units, best_cv_score = select_best_hidden_units(
        X_train_scaled, y_train, hidden_units_list, cv_inner=5
    )
    chosen_units.append(best_units)
    print(f"  → Migliori hidden units: {best_units} (CV score interno: {best_cv_score:.3f})")
    
    # 4. Addestra rete
    net_acc, network, y_pred_net = train_and_evaluate_network(
        X_train_scaled, y_train, X_test_scaled, y_test, best_units
    )
    accuracies_net.append(net_acc)
    print(f"  → Accuratezza rete: {net_acc:.3f}")
    
    # 5. TREPAN
    trepan_clf = TrepanClassifier(
           estimator = mlp, s_min=275, max_internal_nodes=15, random_state=42, base_stumps=MofNTrepanStumpClassifier(max_conditions=4), categorical_features=categorical_features
    )
    trepan_clf.fit(X_train_scaled, y_train)
    y_pred_trepan = trepan_clf.predict(X_test_scaled)  # usa X_test_scaled!
    
    tree_acc = accuracy_score(y_test, y_pred_trepan)
    fidelity = accuracy_score(y_pred_net, y_pred_trepan)
    
    accuracies_tree.append(tree_acc)
    fidelities.append(fidelity)
    
    print(f"  → Accuratezza albero: {tree_acc:.3f}")
    print(f"  → Fedeltà albero-rete: {fidelity:.3f}")
    print()

print("="*60)
print("RISULTATI FINALI (10-fold CV)")
print("="*60)
print(f"Accuratezza rete:     {np.mean(accuracies_net):.3f} ± {np.std(accuracies_net):.3f}")
print(f"Accuratezza albero:   {np.mean(accuracies_tree):.3f} ± {np.std(accuracies_tree):.3f}")
print(f"Fedeltà:              {np.mean(fidelities):.3f} ± {np.std(fidelities):.3f}")
print(f"Hidden units più scelte: {pd.Series(chosen_units).value_counts().to_dict()}")


INIZIO 10-FOLD CV - HEART DISEASE

Fold 1/10
  → Migliori hidden units: 0 (CV score interno: 0.834)
  → Accuratezza rete: 0.871
  → Accuratezza albero: 0.871
  → Fedeltà albero-rete: 1.000

Fold 2/10
  → Migliori hidden units: 0 (CV score interno: 0.834)
  → Accuratezza rete: 0.871
  → Accuratezza albero: 0.871
  → Fedeltà albero-rete: 0.935

Fold 3/10
  → Migliori hidden units: 0 (CV score interno: 0.835)
  → Accuratezza rete: 0.806
  → Accuratezza albero: 0.839
  → Fedeltà albero-rete: 0.968

Fold 4/10
  → Migliori hidden units: 0 (CV score interno: 0.839)
  → Accuratezza rete: 0.833
  → Accuratezza albero: 0.800
  → Fedeltà albero-rete: 0.967

Fold 5/10
  → Migliori hidden units: 0 (CV score interno: 0.835)
  → Accuratezza rete: 0.833
  → Accuratezza albero: 0.833
  → Fedeltà albero-rete: 0.867

Fold 6/10
  → Migliori hidden units: 0 (CV score interno: 0.831)
  → Accuratezza rete: 0.767
  → Accuratezza albero: 0.767
  → Fedeltà albero-rete: 0.933

Fold 7/10
  → Migliori hidden unit

In [19]:
import numpy as np
from sklearn.metrics import accuracy_score
from RuleTree.tree.TrepanClassifier import TrepanClassifier

# 1. Definisci i seed per lo stress test
seeds_da_testare = [10, 42, 111, 123, 999, 11, 86, 78, 123, 22, 52, 66, 88, 101, 222]

# 2. Inizializza le liste per raccogliere le metriche
varianza_accuracy = []
varianza_fidelity = []
varianza_foglie = []

print("Calcolo delle predizioni dell'oracolo (fisse per tutti i seed)...")
# L'oracolo è il modello mlp già addestrato
y_pred_oracolo = mlp.predict(X_test_scaled)

for seed in seeds_da_testare:
    print(f"\nAddestramento albero con Seed: {seed}")
    
    # Inizializza TrepanClassifier con il seed corrente
    trepan_clf = TrepanClassifier(
       estimator = mlp, s_min=275, max_internal_nodes=15, base_stumps=MofNTrepanStumpClassifier(max_conditions=4), random_state=seed, categorical_features=categorical_features
    )
    
    # Esegui il fit sui dati di training
    trepan_clf.fit(X_train_scaled, y_train)
    
    # Ottieni le predizioni dell'albero su X_test_scaled
    y_pred_trepan = trepan_clf.predict(X_test_scaled)
    
    # Calcola l'Accuracy
    acc = accuracy_score(y_test, y_pred_trepan)
    
    # Calcola la Fidelity
    fid = accuracy_score(y_pred_oracolo, y_pred_trepan)
    
    # Calcola il numero di foglie generate
    foglie = len(trepan_clf.get_leaf_nodes())
    
    # Aggiungi i valori alle liste
    varianza_accuracy.append(acc)
    varianza_fidelity.append(fid)
    varianza_foglie.append(foglie)

print("\n" + "="*50)
print(" RISULTATI TEST DI VARIANZA DEI RANDOM SEED")
print("="*50)

print(f"Accuracy : Media {np.mean(varianza_accuracy):.4f} ± Std {np.std(varianza_accuracy):.4f}")
print(f"Fidelity : Media {np.mean(varianza_fidelity):.4f} ± Std {np.std(varianza_fidelity):.4f}")
print(f"Foglie   : Media {np.mean(varianza_foglie):.2f} ± Std {np.std(varianza_foglie):.2f}")

Calcolo delle predizioni dell'oracolo (fisse per tutti i seed)...

Addestramento albero con Seed: 10

Addestramento albero con Seed: 42

Addestramento albero con Seed: 111

Addestramento albero con Seed: 123

Addestramento albero con Seed: 999

Addestramento albero con Seed: 11

Addestramento albero con Seed: 86

Addestramento albero con Seed: 78

Addestramento albero con Seed: 123

Addestramento albero con Seed: 22

Addestramento albero con Seed: 52

Addestramento albero con Seed: 66

Addestramento albero con Seed: 88

Addestramento albero con Seed: 101

Addestramento albero con Seed: 222

 RISULTATI TEST DI VARIANZA DEI RANDOM SEED
Accuracy : Media 0.7911 ± Std 0.0191
Fidelity : Media 0.9711 ± Std 0.0113
Foglie   : Media 14.20 ± Std 2.26
